In [ ]:
#                Chapter 7. Advanced TextGeneration Techniques and Tools

In [ ]:
# `!wget` downloads the Phi-3 Mini 4K Instruct GGUF** model file from the Hugging Face repository and
#   saves it in the current working directory for local use.


In [ ]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

In [ ]:
!mv Phi-3-mini-4k-instruct-fp16.gguf.2 Phi-3-mini-4k-instruct-fp16.gguf

In [ ]:
# Uninstalls the existing versions of the LangChain, LangChain Core, and LangChain
!pip uninstall -y langchain langchain-core langchain-community

In [ ]:
# Installs the the latest versions of LangChain, LangChain Community, and llama-cpp-python
!pip install -U langchain langchain-community llama-cpp-python

In [ ]:
                       #  Loading Quantized Models with LangChain

In [ ]:
#from langchain import LlamaCpp (outdated)
# Make sure the model path is correct for your system!

# importing LlamaCpp class that allows the langchain to run the gguf models using llama.cpp backend
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
model_path="Phi-3-mini-4k-instruct-fp16.gguf",
n_gpu_layers=-1,
max_tokens=500,
n_ctx=2048,
seed=42,
verbose=False
)



In [ ]:
# invoke sends the prompt to the model and return the generated response
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")
# Unfortunately, we get no output! As we have seen in previous chapters,
# Phi 3 requires a specific prompt template

In [ ]:
                    # A Single Link in the Chain: Prompt Template

In [ ]:
# # Import PromptTemplate from LangChain's core prompts module
#from langchain import PromptTemplate
from langchain_core.prompts import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}
<|end|>
<|assistant|>"""

# Create a PromptTemplate object

prompt = PromptTemplate(
    template=template,                # "template" tells LangChain which prompt format to use
    input_variables=["input_prompt"]   # "input_variables" tells LangChain that the template needs a value for "input_prompt"
     )

In [ ]:
# creates a basic LangChain chain by connecting the prompt and llm together.
basic_chain = prompt | llm

In [ ]:
# Use the basic chain to send the input to the prompt template and then to the LLM
basic_chain.invoke(
{
    # Provide the actual question or message for the "input_prompt" variable
"input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
}

                    )

In [ ]:
# Import PromptTemplate from LangChain Core
# PromptTemplate is used to create a reusable prompt with variables
from langchain_core.prompts import PromptTemplate


# Create a prompt template that creates a business name
# {product} is a placeholder that will be replaced with an actual product name
template = """Create a funny name for a business that sells {product}."""


# Create a PromptTemplate object
name_prompt = PromptTemplate(

    # "template" specifies the prompt format we want to use
    template=template,

    # "input_variables" specifies the variable that needs to be provided
    input_variables=["product"]
)

# Connect the prompt template to the language model
# The "|" pipe operator passes the output of name_prompt to llm
# First, the product name is inserted into the prompt
# Then, the completed prompt is sent to the LLM to generate a response
name_chain = name_prompt | llm

# Run the chain by providing "coffee" as the value for the "product" variable
# "coffee" replaces {product} in the prompt
# The final prompt becomes:
# "Create a funny name for a business that sells coffee."

name_chain.invoke({"product": "coffee"})


In [ ]:
                     # A Chain with Multiple Prompts

In [ ]:
#langchain-classic is mainly used to support older LangChain functionality and code.
!pip install -U langchain-classic

In [ ]:
# TITLE

In [ ]:
# from langchain import LLMChain
# LLMChain is used to connect a prompt with a language model (LLM).
from langchain_classic.chains import LLMChain

# Import PromptTemplate to create a reusable prompt with a variable
from langchain_core.prompts import PromptTemplate


# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about
{summary}
. Only return the title.
<|end|>
<|assistant|>"""


title_prompt = PromptTemplate(template=template,   # "template" specifies the prompt format
                              input_variables=["summary"])    # "input_variables" specifies the variable that needs to be provided


# Create an LLMChain by connecting the language model (llm) with the prompt
title = LLMChain(llm=llm,    # specifies the language model that will generate the title

                 # specifies the prompt template used to generate the title
                 prompt=title_prompt,

                 # generated result will be stored with the key "title
                 output_key="title")


#title.invoke({"summary": "a girl that lost her mother"})  JUST FOR CHECKING

In [ ]:
# CHARACTER DESCRIPTION

In [ ]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about
{summary}
 with the
title
{title}
. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt,
output_key="character")


In [ ]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about
{summary}
 with the title
{title}
. The main
character is:
{character}
. Only return the story and it cannot be
longer than one paragraph. <|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
template=template, input_variables=["summary", "title",
"character"]
)
story = LLMChain(llm=llm, prompt=story_prompt,
output_key="story")



In [ ]:
# Combine all three components to create the full chain
llm_chain = title | character | story

llm_chain.invoke("a girl that lost her mother")

In [ ]:
    # Memory: Helping LLMs to Remember Conversations

In [ ]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})



In [ ]:
# Ask the LLM what name was mentioned in the previous message
# This checks whether the LLM can remember the name "Maarten"

basic_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
#Memory: Helping LLMs to Remember Conversations
# Conversation Buffer

In [ ]:
#from langchain import PromptTemplate
from langchain_core.prompts import PromptTemplate

# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:
{chat_history}
{input_prompt}
<|end|>
<|assistant|>"""


# Here, we have two variables:
# 1. "input_prompt" = the user's current question or message
# 2. "chat_history" = the previous conversation that we want the LLM to remember
prompt = PromptTemplate(
template=template,
input_variables=["input_prompt", "chat_history"]
)


In [ ]:
!pip install -U "langchain==0.0.350" "langchain-community==0.0.13"

In [ ]:
# ConversationBufferMemory stores the previous messages in the conversation
from langchain.memory import ConversationBufferMemory

# Import LLMChain to connect the prompt, LLM, and memory together
from langchain.chains import LLMChain

# Define the type of memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
prompt=prompt,
llm=llm,
memory=memory
)

# Run the LLM chain with the user's message
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

In [ ]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
#               Windowed Conversation Buffer

In [ ]:
# This type of memory keeps only a limited number of recent conversations
from langchain.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2,
memory_key="chat_history")

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
prompt=prompt,
llm=llm,
memory=memory
)


# Ask two questions and generate two conversations in its memory

llm_chain.predict(input_prompt="Hi! My name is Maarten and I am 33 years old. What is 1 + 1?")

llm_chain.predict(input_prompt="What is 3 + 3?")

In [ ]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

In [ ]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})

# The LLM indeed has no access to our age since that was not retained in the chat history.

In [ ]:
                    # Conversation Summary

In [ ]:
# Create a prompt template that summarizes the previous conversation
# and combines it with the new conversation lines
summary_prompt_template = """<s><|user|>Summarize the
conversations and update with the new lines.
Current summary:
{summary}
new lines of conversation:
{new_lines}
New summary:<|end|>
<|assistant|>"""


# "new_lines" contains the latest conversation messages
# "summary" contains the existing summary of the previous conversation

summary_prompt = PromptTemplate(
input_variables=["new_lines", "summary"],
template=summary_prompt_template
)


In [ ]:
# Import ConversationSummaryMemory, which summarizes the conversation
from langchain.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
llm=llm,
memory_key="chat_history",
prompt=summary_prompt
)
# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
prompt=prompt,
llm=llm,
memory=memory
)

In [ ]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

In [ ]:
# Check what the summary is thus far
memory.load_memory_variables({})

In [ ]:
             # Agents: Creating a System of LLMs

In [ ]:
#  Install compatable versions

In [ ]:
!pip install -U --force-reinstall \
    "langchain==0.2.16" \
    "langchain-core==0.2.41" \
    "langchain-community==0.2.16" \
    "langchain-openai==0.1.25" \
    "duckduckgo-search"

         Unable to work with open ai it needs billing standards

In [ ]:
# Import the os module to work with environment variables
import os

# Import ChatOpenAI to connect LangChain with an OpenAI
from langchain_openai import ChatOpenAI

# Load OpenAI's LLM with LangChain
os.environ["OPENAI_API_KEY"] = "API_KEY"


# Create an OpenAI chat-based language model using LangChain
openai_llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",   #which OpenAI model to use
    temperature=0
)

print("OpenAI LLM created successfully")

In [ ]:
# PromptTemplate is used to create a reusable prompt with input variables
from langchain.prompts import PromptTemplate

# Create the ReAct template
# This template tells the AI agent how to think about a question,
# choose a tool, use the tool, observe the result, and then give a final answer
react_template = """Answer the following questions as best you can.
You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action

... (this Thought/Action/Action Input/Observation can repeat N times)

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question:
{input}

Thought:
{agent_scratchpad}
"""

# Create a PromptTemplate object using the ReAct template
prompt = PromptTemplate(
    template=react_template,
    input_variables=[
        # Contains the list of tools available to the agent
        "tools",
         # Contains the names of the available tools
        "tool_names",

        # Contains the user's actual question
        "input",

        # Contains the agent's previous reasoning steps and tool results
        "agent_scratchpad"
    ]
)


In [ ]:
# Import load_tools to load built-in LangChain tools
# Import Tool to create our own custom tool
from langchain.agents import load_tools, Tool

# Import DuckDuckGoSearchResults, which allows the agent to search the web
from langchain_community.tools import DuckDuckGoSearchResults

# Create the DuckDuckGo search tool
# This tool can perform web searches and return the search results
search = DuckDuckGoSearchResults()


# Create a custom LangChain Tool using the DuckDuckGo search function
search_tool = Tool(

     # Give the tool a name that the agent can identify and use
    name="duckduck",

      # Describe what the tool does
    # The agent uses this description to understand when it should use the tool
    description="A web search engine. Use this as a search engine for general queries.",


     # Connect the search tool's run function to the custom Tool
    # When the agent uses "duckduck", it calls search.run()
    func=search.run,
)

# Load the built-in LLM Math tool
# "llm-math" allows the agent to solve mathematical calculations
# The OpenAI LLM is provided so the math tool can use the language model

tools = load_tools(
    ["llm-math"],
    llm=openai_llm
)

# Add the search tool to the math tool
# Now the agent has access to both:
tools.append(search_tool)


In [ ]:
# Import AgentExecutor to run the agent and manage the interaction with tools
# Import create_react_agent to create an agent that follows the ReAct approach
from langchain.agents import AgentExecutor, create_react_agent

# Construct the ReAct agent
agent = create_react_agent(
    openai_llm,
    tools,
    prompt
)


# Create an AgentExecutor to run the ReAct agent

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,

     # displays the agents steps in the output for easier understanding
    verbose=True,

    #helps handle errors when the agent's output cannot be correctly understood  by the executor
    handle_parsing_errors=True
)

print("ReAct agent created successfully")

In [ ]:
# Ask the ReAct agent about the MacBook Pro price
response = agent_executor.invoke(
    {
        "input": """What is the current price of a MacBook Pro in USD?
How much would it cost in EUR if the exchange rate is
0.85 EUR for 1 USD?"""
    }
)

print(response["output"])

In [ ]:
# The above code is not working why because open ai needs billing